# Chapter 44: Object Detection with YOLO
## Multimedia & Computer Vision Learning Hub — ITERA
**Author:** Aji Pamungkas Tri Nurcahyo
**Objective:** Memahami arsitektur deteksi objek satu tahap (YOLO), merumuskan fungsi rugi Complete IoU (CIoU), mengimplementasikan Non-Maximum Suppression (NMS), serta mengintegrasikan pipeline inferensi model YOLO nyata (Ultralytics / ONNX).

### 1. Landasan Teori & Formulasi CIoU Loss
Formulasi kerugian Complete IoU (CIoU):
$$\mathcal{L}_{\text{CIoU}} = 1 - \text{IoU} + \frac{\rho^2(\mathbf{b}, \mathbf{b}^{\text{gt}})}{c^2} + \alpha v$$
di mana konsistensi rasio aspek $v$ dan koefisien $\alpha$ dirumuskan:
$$v = \frac{4}{\pi^2} \left( \arctan\frac{w^{\text{gt}}}{h^{\text{gt}}} - \arctan\frac{w}{h} \right)^2, \quad \alpha = \frac{v}{(1 - \text{IoU}) + v}$$

Protokol evaluasi resmi PASCAL VOC dan MS COCO menggunakan Average Precision (AP) melalui interpolasi 11-titik atau 101-titik integrasi kurva Precision-Recall.

In [ ]:
# 2. Setup Lingkungan & Dual-Engine Check
import math
import numpy as np
import sys

print(f"Python Version: {sys.version}")
print(f"NumPy Version: {np.__version__}")

HAS_YOLO = False
try:
    from ultralytics import YOLO
    HAS_YOLO = True
    print("Ultralytics YOLO Engine: INSTALLED & AVAILABLE")
except ImportError:
    print("INFO: Package 'ultralytics' belum terpasang di host lokal.")
    print("Notebook beroperasi dalam Mode Evaluasi Algoritmik Deterministik (CIoU + NMS + 11pt AP).")

In [ ]:
# 3. Implementasi Intersection over Union (IoU) & CIoU Loss
def compute_iou(b1, b2):
    x1 = max(b1[0], b2[0])
    y1 = max(b1[1], b2[1])
    x2 = min(b1[2], b2[2])
    y2 = min(b1[3], b2[3])
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    a1 = (b1[2] - b1[0]) * (b1[3] - b1[1])
    a2 = (b2[2] - b2[0]) * (b2[3] - b2[1])
    union = a1 + a2 - inter
    return inter / union if union > 0 else 0.0

def compute_ciou_loss(b_pred, b_gt):
    iou = compute_iou(b_pred, b_gt)
    cx_p, cy_p = (b_pred[0] + b_pred[2]) / 2, (b_pred[1] + b_pred[3]) / 2
    cx_g, cy_g = (b_gt[0] + b_gt[2]) / 2, (b_gt[1] + b_gt[3]) / 2
    rho2 = (cx_p - cx_g)**2 + (cy_p - cy_g)**2
    c_x1, c_y1 = min(b_pred[0], b_gt[0]), min(b_pred[1], b_gt[1])
    c_x2, c_y2 = max(b_pred[2], b_gt[2]), max(b_pred[3], b_gt[3])
    c2 = (c_x2 - c_x1)**2 + (c_y2 - c_y1)**2
    w_p, h_p = max(1e-6, b_pred[2] - b_pred[0]), max(1e-6, b_pred[3] - b_pred[1])
    w_g, h_g = max(1e-6, b_gt[2] - b_gt[0]), max(1e-6, b_gt[3] - b_gt[1])
    atan_diff = math.atan(w_g / h_g) - math.atan(w_p / h_p)
    v = (4.0 / (math.pi**2)) * (atan_diff**2)
    alpha = v / ((1.0 - iou) + v + 1e-7)
    return 1.0 - (iou - (rho2 / c2) - (alpha * v))

b_gt = np.array([20.0, 20.0, 100.0, 100.0])
b_pred = np.array([40.0, 40.0, 120.0, 120.0])
iou_val = compute_iou(b_pred, b_gt)
ciou_val = compute_ciou_loss(b_pred, b_gt)
print(f"IoU: {iou_val:.4f} (Ekspektasi: 0.3913)")
print(f"CIoU Loss: {ciou_val:.4f} (Ekspektasi: 0.6487)")
assert np.isclose(iou_val, 9.0 / 23.0, atol=1e-4)
assert np.isclose(ciou_val, 0.6487, atol=1e-3)

In [ ]:
# 4. Implementasi Non-Maximum Suppression (NMS)
def nms(boxes, scores, iou_thresh=0.5):
    order = np.argsort(scores)[::-1]
    keep = []
    while len(order) > 0:
        idx = order[0]
        keep.append(idx)
        if len(order) == 1:
            break
        rest = order[1:]
        ious = np.array([compute_iou(boxes[idx], boxes[r]) for r in rest])
        order = rest[ious < iou_thresh]
    return keep

boxes_test = np.array([
    [40.0, 40.0, 120.0, 120.0],  # Box 0: Confidence 0.92
    [42.0, 39.0, 118.0, 121.0],  # Box 1: Overlap tinggi dengan Box 0
    [200.0, 200.0, 280.0, 280.0] # Box 2: Objek independen terpisah
])
scores_test = np.array([0.92, 0.78, 0.85])
kept = nms(boxes_test, scores_test, iou_thresh=0.5)
print(f"Indeks Kotak yang Dipertahankan oleh NMS: {kept}")
assert kept == [0, 2], "Box 1 harus ditekan oleh NMS"

In [ ]:
# 5. Kalkulasi 11-Point Interpolated Average Precision
def compute_11pt_ap(recalls, precisions):
    ap = 0.0
    for r in np.linspace(0.0, 1.0, 11):
        match = precisions[recalls >= r]
        p = np.max(match) if len(match) > 0 else 0.0
        ap += p / 11.0
    return ap

rec = np.array([0.1, 0.2, 0.4, 0.6, 0.8, 1.0])
prec = np.array([1.0, 1.0, 0.8, 0.8, 0.5, 0.4])
calc_ap = compute_11pt_ap(rec, prec)
print(f"Calculated 11-Point AP: {calc_ap:.4f}")
assert np.isclose(calc_ap, 0.7000)

In [ ]:
# 6. Real YOLO Model Inference Pipeline (Ultralytics Specification)
if HAS_YOLO:
    # Load pretrained YOLOv8n (Nano, ~3.2M params, pretrained on MS COCO)
    model_yolo = YOLO('yolov8n.pt')
    print("Official YOLOv8 Nano Model Successfully Loaded.")
    # Buat kanvas citra sintetis uji (3 kanal, 640x640)
    dummy_img = np.zeros((640, 640, 3), dtype=np.uint8)
    dummy_img[200:400, 200:400] = [255, 255, 255] # Kotak terang
    results = model_yolo(dummy_img, verbose=False)
    print("YOLO Real Inference Completed Successfully:")
    print(f"Inference Latency: {results[0].speed['inference']:.2f} ms")
else:
    print("Panduan Eksekusi Model YOLO Riil (Ultralytics AGPL-3.0):")
    print("1. Install Ultralytics: pip install ultralytics")
    print("2. Inisialisasi Model: model = YOLO('yolov8n.pt')  # Otomatis mengunduh bobot COCO 6.2MB")
    print("3. Jalankan Prediksi: results = model('path/to/image.jpg', conf=0.25, iou=0.45)")
    print("4. Akses Output: boxes = results[0].boxes.xyxy (koordinat x1, y1, x2, y2)")
    print("5. Performa: YOLOv8n menghasilkan ~80 FPS pada GPU RTX dan ~25 FPS pada CPU modern.")

### 7. Analisis Hasil, Evaluasi & Rekomendasi
1. **Fungsi Rugi Geometri CIoU**: Mengatasi titik buta IoU standar dengan mengintegrasikan penalti jarak pusat dan konsistensi rasio aspek, mempercepat konvergensi bounding box secara signifikan.
2. **Penyaringan Duplikasi NMS**: Menyaring kotak deteksi berlebih yang berasal dari grid bertetangga secara deterministik berdasarkan confidence score.
3. **Evaluasi Standar COCO mAP**: Menggunakan ambang batas berjenjang $[0.50 : 0.95]$ untuk mengukur kualitas ketepatan kotak pembatas pada tingkat presisi tinggi.